# **4. Zero-shot LLM Inference**

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pandas

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/commonsenseqa_parpahrase_evaluation"

eval_df = pd.read_csv(f"{BASE_PATH}/eval_df.csv")
print(eval_df.shape)

In [ ]:
def save_df(df, filename):
    path = os.path.join(BASE_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

In [ ]:
exp1_df = pd.read_csv(f"{BASE_PATH}/exp1_df.csv")
exp1_df_semantic = pd.read_csv(f"{BASE_PATH}/exp1_df_semantic.csv")
exp2_df = pd.read_csv(f"{BASE_PATH}/exp2_df.csv")
exp2_df_semantic = pd.read_csv(f"{BASE_PATH}/exp2_df_semantic.csv")

### Model Map

In [ ]:
MODEL_MAP = {
    "llama2_7b": "meta-llama/Llama-2-7b-chat-hf",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.1",
    "yi_6b": "01-ai/Yi-6B-Chat",
}

### Prompt Builder

In [ ]:
# Set up prompt for zero-shot models
def build_chat_prompt(question_text, choices):
    choice_lines = "\n".join([f"{k}. {v}" for k, v in choices.items()])

    return f"""Answer the following multiple-choice question.

Question:
{question_text}

Choices:
{choice_lines}

Reply with only one capital letter: A, B, C, D, or E.
"""

In [ ]:
# Ensure only multiple choice answer (A, B, C, D, or E)
def extract_answer_letter(text):
    if text is None:
      return None
    text = text.strip().upper()

    # Best case: single letter answer
    if text in ["A","B","C","D","E"]:
        return text

    # Find letter near beginning
    import re
    m = re.search(r"\b([A-E])\b", text)
    if m:
        return m.group(1)

    return None

### Load Model/Tokenizer

In [ ]:
# Reduce GPU memory Usage (4-bit Quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load models - tokenizer
def load_model_and_tokenizer(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token


    model = AutoModelForCausalLM.from_pretrained(
         model_id,
         quantization_config=bnb_config,
         device_map="auto",
         low_cpu_mem_usage=True
        )
    # Model evaluation mode
    model.eval()
    return tokenizer, model

### One Answer Generation

In [ ]:
# Generate answer from chosen models
def generate_model_answer(model, tokenizer, prompt, max_new_tokens=8):
    messages = [
        {"role": "system", "content": "You answer multiple-choice questions. Reply with only one letter: A, B, C, D, or E."},
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # only decode newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return decoded

### Evaluation Loop for One Model

In [ ]:
# Evaluation loop for one model - Loads model, run questions through model, stores predictions,
# checks for correct answer and returns results as a dataframe
def run_eval_for_model(model_label, model_id, df_to_run):
    print(f"\nLoading {model_label}: {model_id}")
    tokenizer, model = load_model_and_tokenizer(model_id)

    results = []

    for idx, row in df_to_run.iterrows():
        prompt = build_chat_prompt(row["question_text"], row["choices"])
        raw_output = generate_model_answer(model, tokenizer, prompt)
        pred = extract_answer_letter(raw_output)

        results.append({
            "model": model_label,
            "model_id": model_id,
            "id": row["id"],
            "variant_name": row["variant_name"],
            "paraphrase_level": row["paraphrase_level"],
            "question_text": row["question_text"],
            "gold_answer": row["answerKey"],
            "raw_output": raw_output,
            "predicted_answer": pred,
            "correct": pred == row["answerKey"] if pred is not None else False
        })

        if (idx + 1) % 25 == 0:
            print(f"Processed {idx + 1} / {len(df_to_run)}")

    # cleanup
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return pd.DataFrame(results)

### Run Zero-Shot Inference

In [ ]:
results_llama_exp1 = run_eval_for_model("llama2_7b", MODEL_MAP["llama2_7b"], exp1_df)
results_yi_exp1 = run_eval_for_model("yi_6b", MODEL_MAP["yi_6b"], exp1_df)
results_mis_exp1 = run_eval_for_model("mistral_7b", MODEL_MAP["mistral_7b"], exp1_df)

results_exp1 = pd.concat([results_llama_exp1, results_yi_exp1, results_mis_exp1])

save_df_to_drive(results_exp1, "results_exp1_raw.csv")

In [ ]:
results_llama_exp1_sem = run_eval_for_model("llama2_7b", MODEL_MAP["llama2_7b"], exp1_df_semantic)
results_yi_exp1_sem = run_eval_for_model("yi_6b", MODEL_MAP["yi_6b"], exp1_df_semantic)
results_mis_exp1_sem = run_eval_for_model("mistral_7b", MODEL_MAP["mistral_7b"], exp1_df_semantic)

results_exp1_semantic = pd.concat([results_llama_exp1_sem, results_yi_exp1_sem, results_mis_exp1_sem])

save_df_to_drive(results_exp1_semantic, "results_exp1_semantic_raw.csv")

In [ ]:
results_llama_exp2 = run_eval_for_model("llama2_7b", MODEL_MAP["llama2_7b"], exp2_df)
results_yi_exp2  = run_eval_for_model("yi_6b", MODEL_MAP["yi_6b"], exp2_df)
results_mis_exp2 = run_eval_for_model("mistral_7b", MODEL_MAP["mistral_7b"], exp2_df)

results_exp2 = pd.concat([results_llama_exp2, results_yi_exp2, results_mis_exp2])

save_df_to_drive(results_exp2, "results_exp2_raw.csv")

In [ ]:
results_llama_exp2_semantic = run_eval_for_model("llama2_7b", MODEL_MAP["llama2_7b"], exp2_df_semantic)
results_yi_exp2_semantic = run_eval_for_model("yi_6b", MODEL_MAP["yi_6b"], exp2_df_semantic)
results_mis_exp2_semantic = run_eval_for_model("mistral_7b", MODEL_MAP["mistral_7b"], exp2_df_semantic)


results_exp2_semantic = pd.concat([results_llama_exp2_semantic, results_yi_exp2_semantic, results_mis_exp2_semantic])

save_df_to_drive(results_exp2_semantic, "results_exp2_semantic_raw.csv")